# 02 · Variables

Arma los **bloques de variables** de las configuraciones que se corren (en el `main`: el `main`, `AC` y A-consigna; en un branch: el branch y el `main`) y guarda cada uno en `exp/pipeline/bloques/`.

| Bloque | Columnas | Qué es |
|---|---:|---|
| `AC` | 760 | Las 152 variables del mes, más su lag y su delta de 1 y 2 meses (el baseline de la consigna) |
| `CURADA` | 120 | Historia curada: `ratio3`, `z4`, `relmax4` y `mes_ratio3` de las 30 variables de `08_variables.csv` |
| `ACH_V` | 60 | Inestabilidad: desvío de 3 meses del nivel y del delta |
| `ACH_Q` | 90 | Quiebre reciente: los 2 últimos meses contra los 2 anteriores, peor delta y caídas en 3 meses |
| `ACH_T` | 90 | Toda la historia: contra el máximo y la media de todas las fotos previas, y meses desde el máximo |
| `ACH_Z` | 60 | Contra el resto del mes: percentil del delta 1 y del delta 2, con el cero fijo |
| `MVH` | 215 | Las 43 combinaciones de Visa y Master, con su lag y su delta de 1 y 2 meses |
| `R` | 258 | Cambios relativos de Tomás (`r1` y `r2`) de los 129 montos |

**Un bloque guardado se reusa solo si tiene la misma firma**: el mismo código, los mismos parámetros y los mismos datos. Si algo cambió, se rehace.

**Controles:** cantidad de columnas; sin fuga (con los datos cortados en un mes, el bloque de ese mes da lo mismo); y la huella bit a bit de cada bloque tiene que ser la de `resultados/pipeline/huellas_bloques.csv` (versionado). Si no lo es, corta: o la máquina da otros números, o se cambió un bloque que ya existía (un cambio a propósito va como bloque nuevo, con otro nombre).

In [1]:
import importlib
import time

import numpy as np
import pandas as pd

import warnings

import parametros as P
import funciones as F

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')   # falso positivo de pandas 2.2 con Python 3.14

importlib.reload(P)
importlib.reload(F)
P.RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Corrida:', P.CORRIDA['nombre'], '·', len(P.CORRIDA['bloques']), 'bloques')

Corrida: rango · 9 bloques


## 1. Datos

In [2]:
t0 = time.time()
d = F.cargar_datos(P)
filas = F.filas_tabla(d, P.MESES_TABLA)
info = {'mes_fila': d.foto_mes[filas], 'clase': d.clase[filas], 'cliente': d.clave[filas] // 100_000}
assert len(filas) == 1_037_551 and (info['clase'] == 'BAJA+2').sum() == 5_042
F.guardar_filas(P, info)
print(f'Datos: {len(d.foto_mes):,} filas hasta {P.MES_TOPE} · tabla: {len(filas):,} filas · '
      f'{time.time() - t0:.0f} s · memoria máxima {F.memoria_gb()} GB')

Datos: 1,286,487 filas hasta 201910 · tabla: 1,037,551 filas · 1 s · memoria máxima 3.4 GB


## 2. Bloques

In [3]:
configs = F.configuraciones(P)
bloques = list(dict.fromkeys(b for c in configs for b in c['bloques']))
productores = list(dict.fromkeys(F.BLOQUE_DE[b] for b in bloques))
archivo = P.RESULTADOS / 'huellas_bloques.csv'
versionadas = pd.read_csv(archivo).set_index('bloque')['huella'].to_dict() if archivo.exists() else {}
MENSAJE = ('Si no se cambió nada, esta máquina da otros números: avisar antes de comparar branches. '
           'Si el cambio es a propósito, va como bloque nuevo, con otro nombre.')
datos = F.datos_de_bloques(P)
for prod in productores:
    detalle = F.detalle_productor(P, prod, datos)
    if F.productor_vigente(P, prod, detalle):
        print(f'{prod}: ya guardado, con la misma firma')
        continue
    t0 = time.time()
    salida = F.PRODUCTORES[prod](d, filas, P)
    for b, (M, nombres) in salida.items():
        assert M.shape == (len(filas), len(nombres)) and M.dtype == np.float32, b
        if b in F.ESPERADO_COLUMNAS:
            assert len(nombres) == F.ESPERADO_COLUMNAS[b], (b, len(nombres))
    F.control_fuga(d, prod, salida, filas, P)
    huellas_prod = {b: F.huella(M) for b, (M, _) in salida.items()}
    distintas = [b for b, h in huellas_prod.items() if b in versionadas and versionadas[b] != h]
    assert not distintas, f'Bloques distintos de los versionados: {distintas}. {MENSAJE}'
    for b, (M, nombres) in salida.items():
        F.guardar_bloque(P, b, M, nombres)
    F.guardar_firma_productor(P, prod, detalle, huellas_prod)
    iguales = sum(b in versionadas for b in huellas_prod)
    print(f'{prod}: {", ".join(f"{b} ({len(n)})" for b, (_, n) in salida.items())} · sin fuga · '
          f'{iguales} de {len(huellas_prod)} iguales a los versionados · '
          f'{time.time() - t0:.0f} s · memoria máxima {F.memoria_gb()} GB', flush=True)
    del salida

AC: ya guardado, con la misma firma
CURADA: ya guardado, con la misma firma
ACH_4X: ya guardado, con la misma firma
MVH: ya guardado, con la misma firma
R: ya guardado, con la misma firma
RANGO: RANGO (129) · sin fuga · 0 de 1 iguales a los versionados · 6 s · memoria máxima 6.0 GB


## 3. Huellas

La huella es un resumen bit a bit de cada bloque. Queda en `resultados/pipeline/huellas_bloques.csv` (versionado): si otro integrante corre el pipeline en su máquina, tiene que obtener las mismas. Un bloque nuevo se agrega al archivo.

In [4]:
huellas = pd.DataFrame([{'bloque': b, 'columnas': len(F.leer_bloque(P, b)[1]), 'huella': F.huella_bloque(P, b)}
                        for b in bloques])
distintas = [b for b, h in zip(huellas['bloque'], huellas['huella']) if b in versionadas and versionadas[b] != h]
assert not distintas, f'Bloques guardados distintos de los versionados: {distintas}. {MENSAJE}'
if archivo.exists():
    previas = pd.read_csv(archivo)
    huellas = pd.concat([previas[~previas['bloque'].isin(huellas['bloque'])], huellas], ignore_index=True)
huellas.to_csv(archivo, index=False)
print(f'{sum(b in versionadas for b in bloques)} de {len(bloques)} bloques iguales a los versionados')
huellas

8 de 9 bloques iguales a los versionados


,bloque,columnas,huella
0,AC,760,955c386c90580edbe8934d8260da4903
1,CURADA,120,7a304da5333ade4dcb6a4391cf21f164
2,ACH_V,60,566c26e7b6a2bbd302ab522360a56450
3,ACH_Q,90,c3b5a0d9a2fd3e9c27c8c1a0510c3cba
4,ACH_T,90,0bf9ab4209286cc35551629d43f8d27c
5,ACH_Z,60,0d5b5bbff416a883d9331ec846af31b9
6,MVH,215,c733fb853a4cf9bd18f7ce710e5e0cd6
7,R,258,317fbab9bc1a2cd8f394fe33206de956
8,RANGO,129,455de8ea0507ac18919b5ebb7874b98d


## 4. Columnas de cada configuración

In [5]:
for c in configs:
    n = sum(len(F.leer_bloque(P, b)[1]) for b in c['bloques'])
    print(f'{c["nombre"]}: {n:,} columnas')
    if c['bloques'] == P.BLOQUES_PAQ_P3:
        assert n == 1_653
    if c['bloques'] == P.BLOQUES_AC:
        assert n == 760

rango: 1,782 columnas
main: 1,653 columnas
